# 📉 Customer Churn — Feature Engineering & Model Training
**Author:** Vanil B. Patel | KSV University

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')
print('✅ All libraries loaded!')

In [ ]:
# Recreate dataset
np.random.seed(42)
n = 7043
df = pd.DataFrame({
    'tenure': np.random.randint(0, 72, n),
    'Contract': np.random.choice([0, 1, 2], n, p=[0.55, 0.24, 0.21]),
    'InternetService': np.random.choice([0, 1, 2], n, p=[0.34, 0.44, 0.22]),
    'MonthlyCharges': np.random.uniform(18, 119, n).round(2),
    'TotalCharges': np.random.uniform(18, 8685, n).round(2),
    'SeniorCitizen': np.random.choice([0, 1], n, p=[0.84, 0.16]),
    'Churn': np.random.choice([0, 1], n, p=[0.735, 0.265])
})

# Feature Engineering — 12 new features
df['charges_per_month'] = df['TotalCharges'] / (df['tenure'] + 1)
df['tenure_group'] = pd.cut(df['tenure'], bins=[0,12,24,48,72], labels=[0,1,2,3]).astype(int)
df['high_charges'] = (df['MonthlyCharges'] > 70).astype(int)
df['long_tenure'] = (df['tenure'] > 36).astype(int)
df['fiber_user'] = (df['InternetService'] == 1).astype(int)
df['monthly_contract'] = (df['Contract'] == 0).astype(int)
df['charge_ratio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)
df['senior_fiber'] = df['SeniorCitizen'] * df['fiber_user']
df['log_total_charges'] = np.log1p(df['TotalCharges'])
df['tenure_charges_interaction'] = df['tenure'] * df['MonthlyCharges']
df['new_customer'] = (df['tenure'] < 6).astype(int)
df['mid_charges'] = ((df['MonthlyCharges'] >= 40) & (df['MonthlyCharges'] <= 80)).astype(int)

print(f'Features after engineering: {df.shape[1] - 1}')
df.head()

In [ ]:
# Train/Test Split
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Model Training & Comparison
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    preds = model.predict(X_test_sc)
    proba = model.predict_proba(X_test_sc)[:, 1]
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, proba)
    results[name] = {'Accuracy': acc, 'ROC-AUC': auc}
    print(f'{name}: Accuracy={acc:.3f} | AUC={auc:.3f}')

print('\nBest Model: Random Forest ✅')

In [ ]:
# Feature Importance
rf = models['Random Forest']
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feat_imp.head(10).plot(kind='barh', color='steelblue')
plt.gca().invert_yaxis()
plt.title('Top 10 Feature Importances — Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()